In [1]:
import numpy as np
from numba import njit, jit, prange, set_num_threads, float64
import time
from multiprocessing import Pool
from tqdm import tqdm

# Without numba

In [2]:
import numpy as np
from numpy.polynomial import Polynomial
import os
from functools import partial
from scipy.optimize import curve_fit
import math
from copy import deepcopy

c = 299792
v_virial = 250 # km/s

def prepare(xs, ys, freq_diff, freq=None, width=200, start=None, stop=None):
    '''Prepare and optionally clip the spectrum.'''
    ys_mean = ys / ys.mean()
    for i in range(16, len(ys)-16, 16):  # Excise the 4 unstable valley points in each coarse channel 
        ys_mean[i] = np.nan
        ys_mean[i+15] = np.nan
        ys_mean[i+1] = np.nan
        ys_mean[i+14] = np.nan
    
    xs_trimmed = xs
    if freq:
        index = np.argmin(np.abs(xs - freq))
        width = int(width / freq_diff)
        xs_trimmed = xs[index-width:index+width+1]
    else:
        if start or stop:
            start_index = 0 if start is None else np.argmin(np.abs(xs - start))
            stop_index = len(xs) - 1 if stop is None else np.argmin(np.abs(xs - stop))
            xs_trimmed = xs[start_index:stop_index+1]
    
    return xs_trimmed, ys_mean

def normalize_weighted(data, data_dir=None, save_dir=None, a=0.1, b=1, exterior=3, order=5, is_decay=True, **kwargs):
    """
    Standard rolling polynomial normalize procedure. Updated.
    """
    if data_dir:
        xs, ys = np.load(os.path.join(data_dir, data))
    else:
        xs, ys = data
    
    freq_diff = xs[1] - xs[0]
    xs_trimmed, ys_mean = prepare(xs, ys, freq_diff, **kwargs)
    
    new_xs = []
    normalized_spectrum = []
    
    def get_sigma(center):
        if is_decay:
            sigma = v_virial*center/(c*np.sqrt(3))
        else:
            sigma = v_virial*center/(c*np.sqrt(6))
        return sigma
    

    def round_up_to_nearest_odd(number):
        ceiled_number = math.ceil(number)
        return int(ceiled_number + 1) if ceiled_number % 2 == 0 else int(ceiled_number)

    # Loop through the data points and divide each point by the polynomial 
    for x in xs_trimmed:
        i = xs.tolist().index(x)
        window = round_up_to_nearest_odd(2 * exterior * get_sigma(x) / freq_diff)
        
        if i < (window//2): 
            continue
        elif i >= len(xs) - (window//2 + 1):
            break

        ind_xs = np.arange(-(window//2), window//2+1)

        current_ys = ys_mean[i-window//2:i+window//2+1]
        idx = np.isfinite(current_ys)
        
        # Unweighted fit
        #unweighted_params = np.polyfit(ind_xs[idx], current_ys[idx], order)
        unweighted_p = Polynomial.fit(ind_xs[idx], current_ys[idx], order)
        
        # Weighted fit
        sigma = get_sigma(x)
        weights = 1 - a*np.exp(-(freq_diff*ind_xs[idx])**2 / (2*b*sigma**2))
        #weighted_params = np.polyfit(ind_xs[idx], current_ys[idx], order, w=weights)
        weighted_p = Polynomial.fit(ind_xs[idx], current_ys[idx], order, w=weights)

        unweighted = unweighted_p(0) #np.poly1d(unweighted_params)
        weighted = weighted_p(0) #np.poly1d(weighted_params)
        
        new_xs.append(x)
        normalized_spectrum.append(unweighted/weighted)
    
    #new_ys = ys_mean[lower: len(xs)-upper]
    normalized = np.array([new_xs, normalized_spectrum])
    #old = np.array([new_xs, new_ys])
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        np.save(os.path.join(save_dir, data), normalized)
    return normalized

In [4]:
start = time.perf_counter()
_ = normalize_weighted(
        os.listdir('/home/dataadmin/GBTData/SharedDataDirectory/cband_052726/data/preprocessed/1')[0],
        data_dir='/home/dataadmin/GBTData/SharedDataDirectory/cband_052726/data/preprocessed/1'
    )
end = time.perf_counter()
end - start

8.537395761348307

# With numba

In [ ]:
import numpy as np
from numpy.polynomial import Polynomial
import os
from functools import partial
from scipy.optimize import curve_fit
import math
from copy import deepcopy

c = 299792
v_virial = 250 # km/s

@njit
def _coeff_mat(x, deg):
    mat_ = np.zeros(shape=(x.shape[0],deg + 1))
    const = np.ones_like(x)
    mat_[:,0] = const
    mat_[:, 1] = x
    if deg > 1:
        for n in range(2, deg + 1):
            mat_[:, n] = x**n
    return mat_
    
@jit
def _fit_x(a, b):
    # linalg solves ax = b
    det_ = np.linalg.lstsq(a, b)[0]
    return det_
 
@jit
def fit_poly(x, y, deg, w=None):
    a = _coeff_mat(x, deg)
    if w is not None:
        a *= w[:, np.newaxis]
        y *= w
    p = _fit_x(a, y)
    return p

@jit
def eval_polynomial(p, x):
    '''
    Compute polynomial P(x) where P is a vector of coefficients, highest
    order coefficient at P[0].  Uses Horner's Method.
    '''
    result = 0
    for coeff in p[::-1]:
        result = x * result + coeff
    return result

@njit
def normalize_weighted(data, a=0.1, b=1, exterior=3, order=5, is_decay=True, freq=None, width=200, start=None, stop=None):
    """
    Standard rolling polynomial normalize procedure. Updated.
    """
    xs, ys = data
    
    freq_diff = xs[1] - xs[0]
    ys_mean = ys / ys.mean()
    for i in range(16, len(ys)-16, 16):  # Excise the 4 unstable valley points in each coarse channel 
        ys_mean[i] = np.nan
        ys_mean[i+15] = np.nan
        ys_mean[i+1] = np.nan
        ys_mean[i+14] = np.nan
    
    xs_trimmed = xs
    if freq is not None:
        index = np.argmin(np.abs(xs - freq))
        width = int(width / freq_diff)
        xs_trimmed = xs[index-width:index+width+1]
    else:
        if start is not None or stop is not None:
            start_index = 0 if start is None else np.argmin(np.abs(xs - start))
            stop_index = len(xs) - 1 if stop is None else np.argmin(np.abs(xs - stop))
            xs_trimmed = xs[start_index:stop_index+1]
    
    new_xs = []
    normalized_spectrum = []
    
    def get_sigma(center):
        if is_decay:
            sigma = v_virial*center/(c*np.sqrt(3))
        else:
            sigma = v_virial*center/(c*np.sqrt(6))
        return sigma

    def round_up_to_nearest_odd(number):
        ceiled_number = math.ceil(number)
        return int(ceiled_number + 1) if ceiled_number % 2 == 0 else int(ceiled_number)

    # Loop through the data points and divide each point by the polynomial 
    for x in xs_trimmed:
        #i = xs.tolist().index(x)
        i = np.where(xs == x)[0][0]
        window = round_up_to_nearest_odd(2 * exterior * get_sigma(x) / freq_diff)
        
        if i < (window//2): 
            continue
        elif i >= len(xs) - (window//2 + 1):
            break

        ind_xs = np.arange(-(window//2), window//2+1).astype(np.float64)

        current_ys = ys_mean[i-window//2:i+window//2+1]
        idx = np.isfinite(current_ys)
        
        # Unweighted fit
        #unweighted_params = np.polyfit(ind_xs[idx], current_ys[idx], order)
        unweighted_p = fit_poly(ind_xs[idx], current_ys[idx], order)
        
        # Weighted fit
        sigma = get_sigma(x)
        weights = 1 - a*np.exp(-(freq_diff*ind_xs[idx])**2 / (2*b*sigma**2))
        #weighted_params = np.polyfit(ind_xs[idx], current_ys[idx], order, w=weights)
        weighted_p = fit_poly(ind_xs[idx], current_ys[idx], order, w=weights)

        unweighted = eval_polynomial(unweighted_p, 0) #np.poly1d(unweighted_params)
        weighted = eval_polynomial(weighted_p, 0) #np.poly1d(weighted_params)
        
        new_xs.append(x)
        normalized_spectrum.append(unweighted/weighted)
    
    #new_ys = ys_mean[lower: len(xs)-upper]
    normalized = np.array([new_xs, normalized_spectrum])
    #old = np.array([new_xs, new_ys])

    return normalized

In [54]:
# Load in data
data_dir = '/home/dataadmin/GBTData/SharedDataDirectory/cband_052726/data/preprocessed/1'
file = os.listdir(data_dir)[0]
data = np.load(os.path.join(data_dir, file))

# Warm up
_ = normalize_weighted(data, freq=5000, width=10)

start = time.perf_counter()
_ = normalize_weighted(data)
end = time.perf_counter()
end - start

2.317337464541197

In [9]:
xs = data[0]
xs

array([4689.05639648, 4689.23950195, 4689.42260742, ..., 6188.50708008,
       6188.69018555, 6188.87329102], shape=(8192,))

In [10]:
xs[100]

np.float64(4707.366943359375)

In [13]:
np.where(xs == 4707.366943359375)[0][0]

np.int64(100)

In [22]:
weights = 1 - np.exp(-(np.arange(10))**2)
weights

array([0.        , 0.63212056, 0.98168436, 0.99987659, 0.99999989,
       1.        , 1.        , 1.        , 1.        , 1.        ])

In [23]:
weights[:, np.newaxis]

array([[0.        ],
       [0.63212056],
       [0.98168436],
       [0.99987659],
       [0.99999989],
       [1.        ],
       [1.        ],
       [1.        ],
       [1.        ],
       [1.        ]])

# Loop testing

## W/o numba

In [3]:
import numpy as np
from numpy.polynomial import Polynomial
import os
from functools import partial
from scipy.optimize import curve_fit
import math
from copy import deepcopy

c = 299792
v_virial = 250 # km/s

def prepare(xs, ys, freq_diff, freq=None, width=200, start=None, stop=None):
    '''Prepare and optionally clip the spectrum.'''
    ys_mean = ys / ys.mean()
    for i in range(16, len(ys)-16, 16):  # Excise the 4 unstable valley points in each coarse channel 
        ys_mean[i] = np.nan
        ys_mean[i+15] = np.nan
        ys_mean[i+1] = np.nan
        ys_mean[i+14] = np.nan
    
    xs_trimmed = xs
    if freq:
        index = np.argmin(np.abs(xs - freq))
        width = int(width / freq_diff)
        xs_trimmed = xs[index-width:index+width+1]
    else:
        if start or stop:
            start_index = 0 if start is None else np.argmin(np.abs(xs - start))
            stop_index = len(xs) - 1 if stop is None else np.argmin(np.abs(xs - stop))
            xs_trimmed = xs[start_index:stop_index+1]
    
    return xs_trimmed, ys_mean

def normalize_weighted(data, data_dir=None, save_dir=None, a=0.1, b=1, exterior=3, order=5, is_decay=True, **kwargs):
    """
    Standard rolling polynomial normalize procedure. Updated.
    """
    if data_dir:
        xs, ys = np.load(os.path.join(data_dir, data))
    else:
        xs, ys = data
    
    freq_diff = xs[1] - xs[0]
    xs_trimmed, ys_mean = prepare(xs, ys, freq_diff, **kwargs)
    
    new_xs = []
    normalized_spectrum = []
    
    def get_sigma(center):
        if is_decay:
            sigma = v_virial*center/(c*np.sqrt(3))
        else:
            sigma = v_virial*center/(c*np.sqrt(6))
        return sigma
    

    def round_up_to_nearest_odd(number):
        ceiled_number = math.ceil(number)
        return int(ceiled_number + 1) if ceiled_number % 2 == 0 else int(ceiled_number)

    # Loop through the data points and divide each point by the polynomial 
    for x in xs_trimmed:
        i = xs.tolist().index(x)
        window = round_up_to_nearest_odd(2 * exterior * get_sigma(x) / freq_diff)
        
        if i < (window//2): 
            continue
        elif i >= len(xs) - (window//2 + 1):
            break

        ind_xs = np.arange(-(window//2), window//2+1)

        current_ys = ys_mean[i-window//2:i+window//2+1]
        idx = np.isfinite(current_ys)
        
        # Unweighted fit
        #unweighted_params = np.polyfit(ind_xs[idx], current_ys[idx], order)
        unweighted_p = Polynomial.fit(ind_xs[idx], current_ys[idx], order)
        
        # Weighted fit
        sigma = get_sigma(x)
        weights = 1 - a*np.exp(-(freq_diff*ind_xs[idx])**2 / (2*b*sigma**2))
        #weighted_params = np.polyfit(ind_xs[idx], current_ys[idx], order, w=weights)
        weighted_p = Polynomial.fit(ind_xs[idx], current_ys[idx], order, w=weights)

        unweighted = unweighted_p(0) #np.poly1d(unweighted_params)
        weighted = weighted_p(0) #np.poly1d(weighted_params)
        
        new_xs.append(x)
        normalized_spectrum.append(unweighted/weighted)
    
    #new_ys = ys_mean[lower: len(xs)-upper]
    normalized = np.array([new_xs, normalized_spectrum])
    #old = np.array([new_xs, new_ys])
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        np.save(os.path.join(save_dir, data), normalized)
    return normalized

## W/ numba

In [2]:
import numpy as np
from numpy.polynomial import Polynomial
import os
from functools import partial
from scipy.optimize import curve_fit
import math
from copy import deepcopy

c = 299792
v_virial = 250 # km/s

@njit
def _coeff_mat(x, deg):
    mat_ = np.zeros(shape=(x.shape[0],deg + 1))
    const = np.ones_like(x)
    mat_[:,0] = const
    mat_[:, 1] = x
    if deg > 1:
        for n in range(2, deg + 1):
            mat_[:, n] = x**n
    return mat_
    
@jit
def _fit_x(a, b):
    # linalg solves ax = b
    det_ = np.linalg.lstsq(a, b)[0]
    return det_
 
@jit
def fit_poly(x, y, deg, w=None):
    a = _coeff_mat(x, deg)
    if w is not None:
        a *= w[:, np.newaxis]
        y *= w
    p = _fit_x(a, y)
    return p

@jit
def eval_polynomial(p, x):
    '''
    Compute polynomial P(x) where P is a vector of coefficients, highest
    order coefficient at P[0].  Uses Horner's Method.
    '''
    result = 0
    for coeff in p[::-1]:
        result = x * result + coeff
    return result

@njit(cache=True)
def normalize_weighted_numba(data, a=0.1, b=1, exterior=3, order=5, is_decay=True, freq=None, width=200, start=None, stop=None):
    """
    Standard rolling polynomial normalize procedure. Updated.
    """
    xs, ys = data
    
    freq_diff = xs[1] - xs[0]
    ys_mean = ys / ys.mean()
    for i in range(16, len(ys)-16, 16):  # Excise the 4 unstable valley points in each coarse channel 
        ys_mean[i] = np.nan
        ys_mean[i+15] = np.nan
        ys_mean[i+1] = np.nan
        ys_mean[i+14] = np.nan
    
    xs_trimmed = xs
    if freq is not None:
        index = np.argmin(np.abs(xs - freq))
        width = int(width / freq_diff)
        xs_trimmed = xs[index-width:index+width+1]
    else:
        if start is not None or stop is not None:
            start_index = 0 if start is None else np.argmin(np.abs(xs - start))
            stop_index = len(xs) - 1 if stop is None else np.argmin(np.abs(xs - stop))
            xs_trimmed = xs[start_index:stop_index+1]
    
    new_xs = np.zeros(xs_trimmed.shape[0])
    normalized_spectrum = np.zeros(xs_trimmed.shape[0])
    
    def get_sigma(center):
        if is_decay:
            sigma = v_virial*center/(c*np.sqrt(3))
        else:
            sigma = v_virial*center/(c*np.sqrt(6))
        return sigma

    def round_up_to_nearest_odd(number):
        ceiled_number = math.ceil(number)
        return int(ceiled_number + 1) if ceiled_number % 2 == 0 else int(ceiled_number)

    # Loop through the data points and divide each point by the polynomial 
    for j in range(len(xs_trimmed)):
        x = xs_trimmed[j]
        new_xs[j] = x
        #i = xs.tolist().index(x)
        i = np.where(xs == x)[0][0]
        window = round_up_to_nearest_odd(2 * exterior * get_sigma(x) / freq_diff)
        
        if (i < (window//2)) or (i >= len(xs) - (window//2 + 1)): 
            normalized_spectrum[j] = 1
            continue

        ind_xs = np.arange(-(window//2), window//2+1).astype(np.float64)

        current_ys = ys_mean[i-window//2:i+window//2+1]
        idx = np.isfinite(current_ys)
        
        # Unweighted fit
        #unweighted_params = np.polyfit(ind_xs[idx], current_ys[idx], order)
        unweighted_p = fit_poly(ind_xs[idx], current_ys[idx], order)
        
        # Weighted fit
        sigma = get_sigma(x)
        weights = 1 - a*np.exp(-(freq_diff*ind_xs[idx])**2 / (2*b*sigma**2))
        #weighted_params = np.polyfit(ind_xs[idx], current_ys[idx], order, w=weights)
        weighted_p = fit_poly(ind_xs[idx], current_ys[idx], order, w=weights)

        unweighted = eval_polynomial(unweighted_p, 0) #np.poly1d(unweighted_params)
        weighted = eval_polynomial(weighted_p, 0) #np.poly1d(weighted_params)

        normalized_spectrum[j] = unweighted/weighted
        
    #new_ys = ys_mean[lower: len(xs)-upper]
    normalized = np.zeros((2, new_xs.shape[0]))
    normalized[0] = new_xs
    normalized[1] = normalized_spectrum
    #old = np.array([new_xs, new_ys])

    return normalized

In [6]:
# Load in data
data_dir = '/home/dataadmin/GBTData/SharedDataDirectory/cband_052726/data/preprocessed/1'
files = os.listdir(data_dir)[:10]
data = np.array([np.load(os.path.join(data_dir, file)) for file in files])

# w/o numba
start = time.perf_counter()
for dat in data:
    _ = normalize_weighted(dat)
end = time.perf_counter()
print(f'W/o numba: {end-start:.2f} s')

W/o numba: 102.27 s


In [7]:
# w/ numba
# Warm up
_ = normalize_weighted_numba(data[0], freq=5000, width=10)

start = time.perf_counter()
for dat in data:
    _ = normalize_weighted_numba(dat)
end = time.perf_counter()
print(f'W/ numba: {end-start:.2f} s')

W/ numba: 14.36 s


## Parallelized

In [2]:
import numpy as np
from numpy.polynomial import Polynomial
import os
from functools import partial
from scipy.optimize import curve_fit
import math
from copy import deepcopy

c = 299792
v_virial = 250 # km/s

@njit
def _coeff_mat(x, deg):
    mat_ = np.zeros(shape=(x.shape[0],deg + 1))
    const = np.ones_like(x)
    mat_[:,0] = const
    mat_[:, 1] = x
    if deg > 1:
        for n in range(2, deg + 1):
            mat_[:, n] = x**n
    return mat_
    
@jit
def _fit_x(a, b):
    # linalg solves ax = b
    det_ = np.linalg.lstsq(a, b)[0]
    return det_
 
@jit
def fit_poly(x, y, deg, w=None):
    a = _coeff_mat(x, deg)
    if w is not None:
        a *= w[:, np.newaxis]
        y *= w
    p = _fit_x(a, y)
    return p

@jit
def eval_polynomial(p, x):
    '''
    Compute polynomial P(x) where P is a vector of coefficients, highest
    order coefficient at P[0].  Uses Horner's Method.
    '''
    result = 0
    for coeff in p[::-1]:
        result = x * result + coeff
    return result

@njit(parallel=True, cache=True)
def normalize_weighted_numba_parallel(data, a=0.1, b=1, exterior=3, order=5, is_decay=True, freq=None, width=200, start=None, stop=None):
    """
    Standard rolling polynomial normalize procedure. Updated.
    """
    xs, ys = data
    
    freq_diff = xs[1] - xs[0]
    ys_mean = ys / ys.mean()
    for i in range(16, len(ys)-16, 16):  # Excise the 4 unstable valley points in each coarse channel 
        ys_mean[i] = np.nan
        ys_mean[i+15] = np.nan
        ys_mean[i+1] = np.nan
        ys_mean[i+14] = np.nan
    
    xs_trimmed = xs
    if freq is not None:
        index = np.argmin(np.abs(xs - freq))
        width = int(width / freq_diff)
        xs_trimmed = xs[index-width:index+width+1]
    else:
        if start is not None or stop is not None:
            start_index = 0 if start is None else np.argmin(np.abs(xs - start))
            stop_index = len(xs) - 1 if stop is None else np.argmin(np.abs(xs - stop))
            xs_trimmed = xs[start_index:stop_index+1]
    
    new_xs = np.zeros(xs_trimmed.shape[0])
    normalized_spectrum = np.zeros(xs_trimmed.shape[0])
    
    def get_sigma(center):
        if is_decay:
            sigma = v_virial*center/(c*np.sqrt(3))
        else:
            sigma = v_virial*center/(c*np.sqrt(6))
        return sigma

    def round_up_to_nearest_odd(number):
        ceiled_number = math.ceil(number)
        return int(ceiled_number + 1) if ceiled_number % 2 == 0 else int(ceiled_number)

    # Loop through the data points and divide each point by the polynomial 
    for j in prange(len(xs_trimmed)):
        x = xs_trimmed[j]
        new_xs[j] = x
        #i = xs.tolist().index(x)
        i = np.where(xs == x)[0][0]
        window = round_up_to_nearest_odd(2 * exterior * get_sigma(x) / freq_diff)
        
        if (i < (window//2)) or (i >= len(xs) - (window//2 + 1)): 
            normalized_spectrum[j] = 1
            continue

        ind_xs = np.arange(-(window//2), window//2+1).astype(np.float64)

        current_ys = ys_mean[i-window//2:i+window//2+1]
        idx = np.isfinite(current_ys)
        
        # Unweighted fit
        #unweighted_params = np.polyfit(ind_xs[idx], current_ys[idx], order)
        unweighted_p = fit_poly(ind_xs[idx], current_ys[idx], order)
        
        # Weighted fit
        sigma = get_sigma(x)
        weights = 1 - a*np.exp(-(freq_diff*ind_xs[idx])**2 / (2*b*sigma**2))
        #weighted_params = np.polyfit(ind_xs[idx], current_ys[idx], order, w=weights)
        weighted_p = fit_poly(ind_xs[idx], current_ys[idx], order, w=weights)

        unweighted = eval_polynomial(unweighted_p, 0) #np.poly1d(unweighted_params)
        weighted = eval_polynomial(weighted_p, 0) #np.poly1d(weighted_params)

        normalized_spectrum[j] = unweighted/weighted
        
    #new_ys = ys_mean[lower: len(xs)-upper]
    normalized = np.zeros((2, new_xs.shape[0]))
    normalized[0] = new_xs
    normalized[1] = normalized_spectrum
    #old = np.array([new_xs, new_ys])

    return normalized

In [4]:
# Load in data
data_dir = '/home/dataadmin/GBTData/SharedDataDirectory/cband_052726/data/preprocessed/1'
files = os.listdir(data_dir)[:10]
data = np.array([np.load(os.path.join(data_dir, file)) for file in files])

# Warm up

_ = normalize_weighted_numba_parallel(data[0], freq=5000, width=10)

start = time.perf_counter()
for dat in tqdm(data):
    _ = normalize_weighted_numba_parallel(dat)
end = time.perf_counter()
print(f'W/ numba: {end-start:.2f} s')

100%|██████████| 10/10 [00:02<00:00,  4.33it/s]

W/ numba: 2.32 s


## Multiprocessing

In [ ]:
# Load in data
data_dir = '/home/dataadmin/GBTData/SharedDataDirectory/cband_052726/data/preprocessed/1'
files = os.listdir(data_dir)[:10]
data = np.array([np.load(os.path.join(data_dir, file)) for file in files])

# Warm up
_ = normalize_weighted_numba(data[0], freq=5000, width=10)

start = time.perf_counter()
with Pool(processes=10) as p:
    _ = list(p.imap_unordered(normalize_weighted_numba, data))
end = time.perf_counter()
print(f'W/ numba: {end-start:.2f} s')

W/ numba: 11.67 s


In [ ]:
start = time.perf_counter()
with Pool(processes=10) as p:
    _ = list(p.imap_unordered(normalize_weighted, data))
end = time.perf_counter()
print(f'W/o numba: {end-start:.2f} s')

W/o numba: 14.61 s


In [13]:
# Warm up
_ = normalize_weighted_numba(data[0], freq=5000, width=10)

start = time.perf_counter()
_ = normalize_weighted_numba(data[0])
end = time.perf_counter()
print(f'W/ numba: {end-start:.2f} s')

W/ numba: 1.11 s


In [14]:
# Warm up
_ = normalize_weighted_numba(data[0], freq=5000, width=10)

start = time.perf_counter()
with Pool(processes=10) as p:
    _ = list(p.imap_unordered(normalize_weighted_numba, data))
end = time.perf_counter()
print(f'W/ numba: {end-start:.2f} s')

W/ numba: 1.80 s


### Restarting kernel, and trying warm up with no freq/width keywords

In [15]:
# Load in data
data_dir = '/home/dataadmin/GBTData/SharedDataDirectory/cband_052726/data/preprocessed/1'
files = os.listdir(data_dir)[:10]
data = np.array([np.load(os.path.join(data_dir, file)) for file in files])

# Warm up
_ = normalize_weighted_numba(data[0])

start = time.perf_counter()
with Pool(processes=10) as p:
    _ = list(p.imap_unordered(normalize_weighted_numba, data))
end = time.perf_counter()
print(f'W/ numba: {end-start:.2f} s')

W/ numba: 1.57 s


In [18]:
# Load in data
data_dir = '/home/dataadmin/GBTData/SharedDataDirectory/cband_052726/data/preprocessed/1'
files = os.listdir(data_dir)[:10]
data = np.array([np.load(os.path.join(data_dir, file)) for file in files])

# Warm up
_ = normalize_weighted_numba_parallel(data[0])

start = time.perf_counter()
for dat in data:
    _ = normalize_weighted_numba_parallel(dat)
end = time.perf_counter()
print(f'W/ numba: {end-start:.2f} s')

W/ numba: 2.08 s


In [ ]:
# Load in data
data_dir = '/home/dataadmin/GBTData/SharedDataDirectory/cband_052726/data/preprocessed/1'
files = os.listdir(data_dir)[:10]
data = np.array([np.load(os.path.join(data_dir, file)) for file in files])

with Pool(processes=10) as p:
    start = time.perf_counter()
    _ = list(tqdm(p.imap_unordered(normalize_weighted_numba, data), total=len(data)))
    end = time.perf_counter()
print(f'W/ numba: {end-start:.2f} s')

100%|██████████| 10/10 [00:01<00:00,  5.06it/s]


W/ numba: 2.00 s


In [5]:
# Load in data
data_dir = '/home/dataadmin/GBTData/SharedDataDirectory/cband_052726/data/preprocessed/1'
files = os.listdir(data_dir)[:10]
data = np.array([np.load(os.path.join(data_dir, file)) for file in files])

start = time.perf_counter()
for dat in tqdm(data):
    _ = normalize_weighted_numba_parallel(dat)
end = time.perf_counter()
print(f'W/ numba: {end-start:.2f} s')

100%|██████████| 10/10 [00:02<00:00,  4.29it/s]

W/ numba: 2.34 s


In [8]:
v_virial / (c*np.sqrt(3)) * 3

np.float64(0.001444377107768784)

In [10]:
225 / c + (v_virial / (c*np.sqrt(3)) * 3) * (1 + 225 / c)

np.float64(0.0021959815029802903)

In [11]:
0.0021959815029802903 * 5650

12.40729549183864